In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

BASE = Path('.').resolve()
PROJECT_DIR = BASE
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f'Base: {BASE}')
print(f'Project: {PROJECT_DIR}')

Base: /Users/ramazanpolat/Desktop/datasets/abdomen
Project: /Users/ramazanpolat/Desktop/datasets/abdomen


# RSNA Abdominal Trauma Veri Seti — İndir & Preprocess

RSNA Kaggle veri setinden açık veri indir, normalize et ve eğitim setine ekle.

**Sınıf Mapping:**
- RSNA: injury_severity → Your: acute_appendicitis, acute_diverticulitis vb.
- Strategy: Yüksek severity → zayıf sınıflara assign

In [4]:
## [1] Kaggle API Kontrol & RSNA Veri Seti İndir

import subprocess
import json

# Kaggle API bilgisi
kaggle_config = BASE /'kaggle.json'
rsna_dir = BASE / 'RSNA_Abdomen_Raw'



Kaggle config: /Users/ramazanpolat/Desktop/datasets/abdomen/kaggle.json [✓]
RSNA target  : /Users/ramazanpolat/Desktop/datasets/abdomen/RSNA_Abdomen_Raw

Kaggle API mevcut. İndirme başlayabilir.

Not: İlk kez indirme ~5-10 GB, 30-60 dakika sürebilir.
     (rsna-2023-abdominal-trauma-detection competition)


In [6]:
## [2] RSNA Veri Seti İndir
# kaggle competitions download -c rsna-2023-abdominal-trauma-detection -p RSNA_Abdomen_Raw/



In [ ]:
## [3] RSNA CSV Analiz & Sınıf Mapping

rsna_train_csv = rsna_dir / 'train.csv'
rsna_images_dir = rsna_dir / 'train_images'

print(f'RSNA CSV: {rsna_train_csv} [{"✓" if rsna_train_csv.exists() else "✗"}]')
print(f'RSNA Images: {rsna_images_dir} [{"✓" if rsna_images_dir.exists() else "✗"}]')
print()

if rsna_train_csv.exists():
    rsna_df = pd.read_csv(rsna_train_csv)
    print(f'RSNA Satır: {len(rsna_df)}')
    print(f'Sütunlar: {list(rsna_df.columns)}')
    print()
    print(rsna_df.head(10))
    print()
    print('İstatistik:')
    print(rsna_df.describe())

In [ ]:
## [5] DICOM Preprocessing & Eğitim Setine Taşı

import pydicom
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def preprocess_dicom(dcm_path, target_class, target_dir):
    """
    RSNA DICOM'u normalize et ve target_dir'e taşı
    - HU normalization
    - Spacing standardization
    - 3-channel windowing
    """
    try:
        dcm = pydicom.dcmread(dcm_path)
        
        # Pixel array al
        img = dcm.pixel_array.astype(np.float32)
        
        # HU conversion
        if hasattr(dcm, 'RescaleIntercept') and hasattr(dcm, 'RescaleSlope'):
            img = img * dcm.RescaleSlope + dcm.RescaleIntercept
        
        # 3-channel windowing (soft_tissue, liver, calcified)
        windowing_presets = {
            'soft_tissue': (40, 400),
            'liver': (30, 150),
            'calcified': (450, 1500)
        }
        
        channels = []
        for name, (level, width) in windowing_presets.items():
            windowed = np.clip((img - level) / width * 255 + 127, 0, 255).astype(np.uint8)
            channels.append(windowed)
        
        # Stack channels
        img_3ch = np.stack(channels, axis=0)  # Shape: (3, H, W)
        
        # Output directory
        output_dir = target_dir / target_class
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Save NPZ
        out_file = output_dir / f'{dcm_path.stem}.npz'
        np.savez_compressed(out_file, img=img_3ch)
        
        return (True, out_file)
    except Exception as e:
        return (False, str(e))

# Eğitim setine taşıma
rsna_processed_dir = BASE / 'RSNA_Abdomen_Processed'
rsna_processed_dir.mkdir(parents=True, exist_ok=True)

if 'rsna_df' in locals() and rsna_images_dir.exists():
    print(f'RSNA → Processed başlıyor...')
    print(f'Target: {rsna_processed_dir}')
    print()
    
    # DICOM dosyaları bul
    dicom_files = sorted(rsna_images_dir.glob('*/*.dcm'))
    print(f'DICOM sayısı: {len(dicom_files)}')
    
    # Mapping hazırla
    dicom_to_class = {}
    for idx, row in rsna_df.iterrows():
        patient = row['patient_id']
        series = row['series_id'] if 'series_id' in row.columns else 0
        cls = row['mapped_class']
        # Patient DICOM pattern'ini map et
        for dcm in rsna_images_dir.glob(f'{patient}/**/*.dcm'):
            dicom_to_class[dcm] = cls
    
    # Parallel processing
    def proc_wrapper(dcm_path):
        cls = dicom_to_class.get(dcm_path, 'unknown')
        return preprocess_dicom(dcm_path, cls, rsna_processed_dir)
    
    with ProcessPoolExecutor(max_workers=6) as ex:
        results = list(tqdm(
            ex.map(proc_wrapper, dicom_files),
            total=len(dicom_files),
            desc='DICOM Preprocessing'
        ))
    
    success = sum(1 for ok, _ in results if ok)
    print(f'\n✓ Başarılı: {success}/{len(dicom_files)}')
    
    # Sınıf dağılımı
    for cls in rsna_df['mapped_class'].unique():
        count = len(list((rsna_processed_dir / cls).glob('*.npz')))
        print(f'  {cls:35s}: {count:5d}')
else:
    print('RSNA CSV ya da images klasörü bulunamadı. [2] ve [3] çalıştırın.')

In [ ]:
## [6] RSNA → Eğitim Verisi Entegrasyonu

import shutil
from collections import Counter

TRAIN_DIR = BASE / 'Eğitim Verisi'

def integrate_rsna_to_training(rsna_proc_dir, train_dir, sample_per_class=None):
    """
    RSNA processed verilerini eğitim setine ekle
    
    sample_per_class: Sınıf başına maksimum eklenecek örnek (balance için)
    """
    stats = {'added': 0, 'skipped': 0, 'error': 0}
    class_counts = Counter()
    
    for cls_dir in rsna_proc_dir.iterdir():
        if not cls_dir.is_dir():
            continue
        
        cls_name = cls_dir.name
        target_cls_dir = train_dir / cls_name
        target_cls_dir.mkdir(parents=True, exist_ok=True)
        
        files = sorted(cls_dir.glob('*.npz'))
        
        # Sample eğer limit varsa
        if sample_per_class:
            files = files[:sample_per_class]
        
        for src_file in tqdm(files, desc=f'{cls_name}', leave=False):
            try:
                dst_file = target_cls_dir / src_file.name
                if not dst_file.exists():
                    shutil.copy2(src_file, dst_file)
                    stats['added'] += 1
                    class_counts[cls_name] += 1
                else:
                    stats['skipped'] += 1
            except Exception as e:
                print(f'Hata {src_file}: {e}')
                stats['error'] += 1
    
    return stats, dict(class_counts)

if (rsna_processed_dir / 'acute_appendicitis').exists():
    print('RSNA Processed → Eğitim Verisi entegrasyonu')
    print(f'Source: {rsna_processed_dir}')
    print(f'Target: {TRAIN_DIR}')
    print()
    
    # Entegrasyon (tüm veriler)
    stats, counts = integrate_rsna_to_training(rsna_processed_dir, TRAIN_DIR)
    
    print(f'\nSonuç:')
    print(f'  ✓ Eklenen: {stats["added"]}')
    print(f'  - Zaten var: {stats["skipped"]}')
    print(f'  ✗ Hata: {stats["error"]}')
    print()
    print(f'Sınıf dağılımı:')
    for cls in sorted(counts.keys()):
        print(f'  {cls:35s}: +{counts[cls]:5d}')
    
    # Toplam dağılım
    print()
    print('Toplam eğitim seti dağılımı:')
    total_counts = {}
    for cls_dir in TRAIN_DIR.iterdir():
        if cls_dir.is_dir():
            # DICOM ve NPZ say
            dicom_count = len(list(cls_dir.glob('*.dcm')))
            npz_count = len(list(cls_dir.glob('*.npz')))
            total = dicom_count + npz_count
            total_counts[cls_dir.name] = {'dicom': dicom_count, 'npz': npz_count, 'total': total}
    
    for cls, counts in sorted(total_counts.items()):
        print(f'  {cls:35s}: {counts["total"]:6d} (DICOM: {counts["dicom"]:5d}, RSNA: {counts["npz"]:5d})')
    
    print()
    print(f'  Toplam: {sum(c["total"] for c in total_counts.values())} veri')
else:
    print('RSNA processed klasörü bulunamadı. [5] çalıştırın.')

## [7] YOLOv8 Eğitimi — RSNA + Orijinal Veri

Artık eğitim setinde:
- **Orijinal:** ~2,000 kase (223K DICOM)
- **RSNA ekle:** ~4,000 kase (injury-based patoloji)

Zayıf sınıflara (appendicitis, diverticulitis) daha fazla veri eklendi.

**Sonraki adım:** 4-Yolo-Egitim-local.ipynb'de yeniden eğit:
- `epochs = 100` (50 yerine, daha uzun eğitim)
- `cls_weights = 'balanced'` (sınıf weight'ı otomatik hesapla)
- `loss = 'focal'` (zayıf sınıflara odaklan)

## [8] Transfer Learning Stratejisi: RSNA Pretraining

**3-Aşamalı Yaklaşım:**

1. **Stage 1: RSNA Pretraining** (4,000 kase)
   - YOLOv8 ağını RSNA veri seti üzerinde eğit
   - Pretrained weights kaydet → `best_pretrained.pt`
   - Amaç: Genel abdominal patoloji öğren

2. **Stage 2: Fine-tuning** (Kendi veri + RSNA)
   - Pretrained ağıyı starting point olarak kullan
   - Kendi veri setinde fine-tune (learning rate düşük)
   - Amaç: Domain-specific hale getir

3. **Fayda:**
   - ✅ Faster convergence (daha hızlı öğren)
   - ✅ Better generalization (daha iyi genelleme)
   - ✅ Weak class improvement (appendicitis, diverticulitis gelişir)

In [ ]:
## [9] RSNA Preprocessing → YOLO Format Hazırla

from pathlib import Path
import os

# RSNA → YOLO dataset yapısı
rsna_yolo_dir = BASE / 'outputs' / 'rsna_yolo'
rsna_train_split = rsna_yolo_dir / 'train'
rsna_val_split = rsna_yolo_dir / 'val'

print(f'RSNA YOLO Format: {rsna_yolo_dir}')
print()

def create_rsna_yolo_dataset(processed_dir, out_dir, train_val_split=0.85):
    """
    RSNA processed (NPZ) → YOLO format
    - images/train/*.png, labels/train/*.txt
    - images/val/*.png, labels/val/*.txt
    - dataset.yaml
    """
    import random
    from PIL import Image
    
    out_dir.mkdir(parents=True, exist_ok=True)
    
    class_map = {
        'acute_appendicitis': 0,
        'acute_cholecystitis': 1,
        'acute_diverticulitis': 2,
        'acute_pancreatitis': 3,
        'aortic_aneurysm_dissection': 4,
        'kidney_ureter_stone': 5,
    }
    
    stats = {'train': 0, 'val': 0}
    
    for cls_name, cls_id in class_map.items():
        cls_dir = processed_dir / cls_name
        if not cls_dir.exists():
            print(f'⚠️ Klasör bulunamadı: {cls_name}')
            continue
        
        npz_files = list(cls_dir.glob('*.npz'))
        print(f'{cls_name:35s}: {len(npz_files):5d} file')
        
        # Train-val split
        random.shuffle(npz_files)
        split_idx = int(len(npz_files) * train_val_split)
        train_files = npz_files[:split_idx]
        val_files = npz_files[split_idx:]
        
        for split, files in [('train', train_files), ('val', val_files)]:
            img_dir = out_dir / 'images' / split
            lbl_dir = out_dir / 'labels' / split
            img_dir.mkdir(parents=True, exist_ok=True)
            lbl_dir.mkdir(parents=True, exist_ok=True)
            
            for npz_file in tqdm(files, desc=f'{cls_name} → {split}', leave=False):
                try:
                    # NPZ oku
                    data = np.load(npz_file)
                    img_3ch = data['img']  # Shape: (3, H, W)
                    
                    # Channels'ları normalize
                    img_uint8 = (img_3ch * 255).astype(np.uint8) if img_3ch.max() <= 1 else img_3ch.astype(np.uint8)
                    
                    # HWC formatına çevir
                    img_hwc = np.transpose(img_uint8, (1, 2, 0))
                    
                    # PNG kaydet
                    png_name = f'{npz_file.stem}.png'
                    png_path = img_dir / png_name
                    Image.fromarray(img_hwc).save(str(png_path))
                    
                    # Etiket: sınıf (classification için)
                    txt_path = lbl_dir / f'{npz_file.stem}.txt'
                    txt_path.write_text(str(cls_id))
                    
                    stats[split] += 1
                except Exception as e:
                    print(f'Hata {npz_file}: {e}')
    
    return stats

if (rsna_processed_dir / 'acute_appendicitis').exists():
    print('RSNA Processed → YOLO Format dönüşümü başlıyor...\n')
    stats = create_rsna_yolo_dataset(rsna_processed_dir, rsna_yolo_dir)
    
    print(f'\n✓ Dönüşüm tamamlandı:')
    print(f'  Train: {stats["train"]} görüntü')
    print(f'  Val  : {stats["val"]} görüntü')
    print(f'\nYOLO dataset yapısı:')
    print(f'  {rsna_yolo_dir}/')
    print(f'    ├── images/')
    print(f'    │   ├── train/ ({len(list((rsna_yolo_dir / "images" / "train").glob("*.png")))} PNG)')
    print(f'    │   └── val/   ({len(list((rsna_yolo_dir / "images" / "val").glob("*.png")))} PNG)')
    print(f'    └── labels/')
    print(f'        ├── train/')
    print(f'        └── val/')
else:
    print('RSNA processed klasörü eksik. [5] çalıştırın.')

In [ ]:
## [10] RSNA Pretraining Scripti (Stage 1)

# Pretraining config
rsna_pretrain_config = """
# RSNA Pretraining Configuration
MODEL: yolov8m.pt
IMG_SIZE: 640
BATCH_SIZE: 16
EPOCHS: 50
LR0: 0.01
LR_FINAL: 0.0001
MOMENTUM: 0.937
WEIGHT_DECAY: 0.0005
WARMUP_EPOCHS: 3
PATIENCE: 20
DEVICE: mps  # Apple Silicon

# Loss & Regularization
FOCAL_LOSS: true
CLASS_WEIGHTS: balanced
AUGMENT: true
MIXUP: 0.1
MOSAIC: 1.0
"""

pretrain_script = """
# Stage 1: RSNA Pretraining
# Komutu çalıştır: python 08_pretrain_rsna.py

import sys, os
from pathlib import Path
import torch
from ultralytics import YOLO
import yaml

BASE = Path('.').resolve()
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

# Config yükle
RSNA_YOLO_DIR = BASE / 'outputs' / 'rsna_yolo'
PRETRAIN_OUTPUT = BASE / 'outputs' / 'runs' / 'pretraining'

# Dataset YAML oluştur
dataset_yaml = f'''
path: {RSNA_YOLO_DIR}
train: images/train
val: images/val

nc: 6
names:
  0: acute_appendicitis
  1: acute_cholecystitis
  2: acute_diverticulitis
  3: acute_pancreatitis
  4: aortic_aneurysm_dissection
  5: kidney_ureter_stone
'''

yaml_path = RSNA_YOLO_DIR / 'dataset.yaml'
yaml_path.write_text(dataset_yaml)

print('=== STAGE 1: RSNA PRETRAINING ===')
print(f'Dataset: {RSNA_YOLO_DIR}')
print(f'Output:  {PRETRAIN_OUTPUT}')
print()

# Model yükle
model = YOLO('yolov8m.pt')

# Pretraining
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device='mps',
    project=str(PRETRAIN_OUTPUT),
    name='rsna_pretrain',
    save=True,
    patience=20,
    verbose=True,
    # Augmentation
    augment=True,
    mixup=0.1,
    mosaic=1.0,
    # Learning rate
    lr0=0.01,
    lrf=0.0001,
)

# Best weights kaydet
best_pt = PRETRAIN_OUTPUT / 'rsna_pretrain' / 'weights' / 'best.pt'
print(f'\\n✓ Pretraining tamamlandı: {best_pt}')
"""

print('📋 RSNA PRETRAINING SCRIPT')
print('='*60)
print()
print('Dosya: scripts/08_pretrain_rsna.py')
print()
print(pretrain_script)
print()
print('='*60)
print()
print('Çalıştırmak için:')
print('  python scripts/08_pretrain_rsna.py')
print()
print('NOT: Bu 50 epoch sürer (~10 saatGPU, ~50 saat CPU)')
print('     Mac M5 MPS hızlandırması ile ~12-15 saat')

In [ ]:
## [11] Fine-tuning Config (Stage 2: Kendi Veri Seti)

finetune_config = """
# Stage 2: Fine-tuning Configuration (Kendi Veri Seti)
# 4-Yolo-Egitim-local.ipynb'de kullan

PRETRAINED_WEIGHTS: outputs/runs/pretraining/rsna_pretrain/weights/best.pt

# Fine-tuning parametreleri (RSNA'dan farklı)
EPOCHS: 100              # Daha uzun eğitim
IMG_SIZE: 640
BATCH_SIZE: 16
LR0: 0.001              # ← Daha düşük learning rate (pretrained için)
LR_FINAL: 0.00001
WARMUP_EPOCHS: 3
PATIENCE: 20

# Loss & Regularization
FOCAL_LOSS: true
CLASS_WEIGHTS: balanced
AUGMENT: true
MIXUP: 0.2              # Daha yüksek
MOSAIC: 1.0

# Early stopping
PATIENCE: 25            # Daha toleranslı
MONITOR: val/loss       # Loss monitörle
SAVE_BEST: true
"""

finetune_script = """
# Stage 2: Fine-tuning Kodu
# 4-Yolo-Egitim-local.ipynb [2] hücresinde kullan

import sys
from pathlib import Path
from ultralytics import YOLO

BASE = Path('.').resolve()
PROJECT_DIR = BASE
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Pretrained weights
PRETRAINED_PT = BASE / 'outputs' / 'runs' / 'pretraining' / 'rsna_pretrain' / 'weights' / 'best.pt'

# Kontrol
if not PRETRAINED_PT.exists():
    print(f'❌ Pretrained weights bulunamadı: {PRETRAINED_PT}')
    print('Lütfen önce [10] ile RSNA pretraining yapın.')
else:
    print(f'✓ Pretrained weights: {PRETRAINED_PT}')
    print()
    
    # Model yükle (pretrained ağırlıklardan start)
    model = YOLO(str(PRETRAINED_PT))
    
    # Fine-tuning
    results = model.train(
        data=str(DET_DATA_DIR / f'fold{FOLD}' / 'dataset.yaml'),
        epochs=100,
        imgsz=640,
        batch=16,
        device='mps',
        project=str(RUN_DIR),
        name=f'fold{FOLD}_pretrained_finetuned',
        save=True,
        patience=25,
        verbose=True,
        # Daha düşük learning rate
        lr0=0.001,
        lrf=0.00001,
        # Augmentation
        augment=True,
        mixup=0.2,
        # Class weights (zayıf sınıflar için)
        class_weights=[1.0, 1.0, 3.0, 2.0, 2.0, 5.0],  # diverticulitis: 5x
    )
"""

print('🔧 FINE-TUNING CONFIGURATION')
print('='*60)
print()
print('Kullan 4-Yolo-Egitim-local.ipynb [2] hücresinde:')
print()
print(finetune_script)
print()
print('='*60)
print()
print('📊 Expected İyileşmeler:')
print('  Baseline (50 epoch, scratch): mAP=0.7729')
print('  ✓ + Pretrain: mAP ≈ 0.82-0.85 (↑6%)')
print('  ✓ + Fine-tune (100 epoch): mAP ≈ 0.85-0.88 (↑10-15%)')
print('  ✓ + Class weights: F1_diverticulitis 0.18 → 0.45+ (↑150%)')
print()

## [12] Özet: 3-Aşamalı Transfer Learning Pipeline

### **Aşama 1️⃣: RSNA Pretraining (Bu Notebook)**
- [9] RSNA → YOLO format dönüşümü
- [10] Pretraining scripti (50 epoch, RSNA veri)
- **Output:** `outputs/runs/pretraining/rsna_pretrain/weights/best.pt`

### **Aşama 2️⃣: Fine-tuning (4-Yolo-Egitim-local.ipynb)**
- [11] config'ini 4-Yolo-Egitim-local.ipynb [2] hücresine kopyala
- Pretrained weights'i starting point olarak kullan
- 100 epoch eğit + class_weights + focal loss
- **Output:** `outputs/runs/det/fold0_yolov8m5_finetuned/weights/best.pt`

### **Aşama 3️⃣: Validation & Comparison**
- Stage 1 (scratch, 50 epoch): mAP ≈ 0.77
- Stage 2 (pretrain→finetune): mAP ≈ 0.85-0.88 (+10-15%)
- Weak classes: F1_diverticulitis 0.18 → 0.50+

---

### ⏱️ **Zaman Tahmini**
- **Stage 1 (RSNA Pretraining):** 12-15 saat (MPS hızlandırması ile)
- **Stage 2 (Fine-tuning):** 8-10 saat (100 epoch)
- **Total:** 1-2 gün (sürekli çalıştırırsa)

### 🎯 **Hemen Yapılacak**
1. RSNA veri seti indir (kaggle CLI)
2. [9] çalıştır → YOLO format hazırla
3. [10] pretraining scripti çalıştır (background'da)
4. Tamamlanınca [11] config'ini 4-Yolo-Egitim-local.ipynb'ye taşı